# SafeAd - Face Age Analysis Colab Notebook

This notebook demonstrates how to run the face detection and age estimation module in Google Colab.

In [ ]:
!pip install torch torchvision facenet-pytorch transformers pillow matplotlib numpy

In [ ]:
import torch
import sys
print('Python Version:', sys.version)
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
os.makedirs('ai/face_age', exist_ok=True)


In [ ]:
%%writefile ai/face_age/__init__.py


In [ ]:
%%writefile ai/face_age/model_manager.py
import torch
import gc

class ModelManager:
    @staticmethod
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    @staticmethod
    def clear_memory():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
%%writefile ai/face_age/utils.py
def normalize_age_prediction(age_range: str, confidence: float) -> str:
    if confidence < 0.5:
        return "UNKNOWN"
    
    # Simple mapping based on expected ranges from the ViT model
    child_ranges = ["0-2", "3-9"]
    teen_ranges = ["10-19"]
    
    if age_range in child_ranges:
        return "CHILD"
    elif age_range in teen_ranges:
        return "TEEN"
    else:
        return "ADULT"


In [ ]:
%%writefile ai/face_age/face_detector.py
import torch
from facenet_pytorch import MTCNN
from PIL import Image
import numpy as np

from .model_manager import ModelManager

class FaceDetector:
    def __init__(self, device=None, keep_all=True):
        self.device = device or ModelManager.get_device()
        self.detector = MTCNN(keep_all=keep_all, device=self.device)

    def detect_faces(self, image: Image.Image):
        # Convert to RGB if necessary
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        boxes, probs = self.detector.detect(image)
        
        faces = []
        if boxes is not None:
            for i, (box, prob) in enumerate(zip(boxes, probs)):
                if prob is not None:
                    # MTCNN boxes are [x1, y1, x2, y2]
                    x1, y1, x2, y2 = [int(b) for b in box]
                    # Crop face
                    face_crop = image.crop((x1, y1, x2, y2))
                    faces.append({
                        "face_id": i + 1,
                        "bbox": [x1, y1, x2, y2],
                        "detection_confidence": float(prob),
                        "face_crop": face_crop
                    })
        return faces


In [ ]:
%%writefile ai/face_age/age_estimator.py
import torch
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch.nn.functional as F

from .model_manager import ModelManager

class AgeEstimator:
    def __init__(self, model_name="nateraw/vit-age-classifier", device=None):
        self.device = device or ModelManager.get_device()
        self.feature_extractor = ViTImageProcessor.from_pretrained(model_name)
        self.model = ViTForImageClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def estimate_age(self, face_image: Image.Image):
        inputs = self.feature_extractor(images=face_image, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)
            
            confidence, predicted_class_idx = torch.max(probs, 1)
            predicted_class_idx = predicted_class_idx.item()
            confidence = confidence.item()
            
            age_range = self.model.config.id2label[predicted_class_idx]
            
        return {
            "estimated_age": None,
            "age_range": age_range,
            "confidence": round(confidence, 4)
        }


In [ ]:
%%writefile ai/face_age/face_age_pipeline.py
from PIL import Image
from typing import Dict, Any

from .face_detector import FaceDetector
from .age_estimator import AgeEstimator
from .model_manager import ModelManager
from .utils import normalize_age_prediction

class FaceAgePipeline:
    def __init__(self):
        # Lazy loading of models
        self.detector = None
        self.age_estimator = None

    def _initialize_models(self):
        if self.detector is None:
            self.detector = FaceDetector()
        if self.age_estimator is None:
            self.age_estimator = AgeEstimator()

    def analyze(self, image: Image.Image) -> Dict[str, Any]:
        self._initialize_models()
        
        detected_faces = self.detector.detect_faces(image)
        
        result_faces = []
        for face in detected_faces:
            age_estimation = self.age_estimator.estimate_age(face["face_crop"])
            
            normalized_group = normalize_age_prediction(
                age_range=age_estimation["age_range"], 
                confidence=age_estimation["confidence"]
            )
            
            # don't return face_crop in final JSON output
            result_faces.append({
                "face_id": face["face_id"],
                "bbox": face["bbox"],
                "detection_confidence": round(face["detection_confidence"], 4),
                "age_estimation": age_estimation,
                "normalized_age_group": normalized_group
            })
            
        ModelManager.clear_memory()
            
        return {
            "media_type": "image",
            "faces_detected": len(result_faces),
            "faces": result_faces
        }


NOTE: The code cells above dynamically generate the Python modules inside the Colab environment so you don't need to clone a GitHub repository!

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import json

from ai.face_age.face_age_pipeline import FaceAgePipeline

# Initialize Pipeline (loads lazily)
pipeline = FaceAgePipeline()
print('Pipeline Initialized')

In [ ]:
# Upload an image via colab interface (or manually provide a test image)
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0] if uploaded else None

In [ ]:
if image_path:
    img = Image.open(image_path)
    
    # Run Analysis
    result = pipeline.analyze(img)
    
    # Save JSON result
    with open('result.json', 'w') as f:
        json.dump(result, f, indent=4)
    print(json.dumps(result, indent=4))
    
    # Visualization
    fig, ax = plt.subplots(1)
    ax.imshow(img)
    
    for face in result['faces']:
        x1, y1, x2, y2 = face['bbox']
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        
        age_range = face['age_estimation']['age_range']
        confidence = face['age_estimation']['confidence']
        if confidence < 0.5:
            label = f"Face {face['face_id']}\nAge: Unknown"
        else:
            label = f"Face {face['face_id']}\nAge: {age_range}\nConf: {confidence:.2f}"
            
        ax.text(x1, y1 - 10, label, color='white', backgroundcolor='red', fontsize=10)
    
    plt.savefig('annotated_image.jpg')
    plt.show()